# Mini Test30 — đánh giá checkpoint RL-RC-DoT/VVenC adapter/3

**Chỉ chạy SAU Mini Trial 20.480** khi đã chọn checkpoint tốt nhất trên Val4. Input Kaggle cần BDD100K MOV và gói `*_KAGGLE_SAFE.zip` (hoặc thư mục giải nén có `session_payload.bin`) xuất từ **Mini Trial**. Notebook khóa đúng source và kiểm tra giao thức trước khi mã hóa.

- Test30 khác Train16 và Val4, cố định theo seed; mỗi video dùng **256 frame đầu**, 3 bitrate **50/120/200 kbps**; hai chế độ mã hóa **Anchor & RL Agent**. Tổng 90 cặp nếu chạy đầy đủ.
- Mỗi session mặc định chỉ chạy thêm **15 cặp** rồi xuất evaluation SAFE để resume phiên sau (giảm rủi ro quá giới hạn Kaggle). Kết quả Test30 là **khảo sát**; không dùng để điều chỉnh thuật toán. Mini 3 bitrate không báo BD-rate 5 điểm.
- **Không dùng gói Full65/Val7 hay adapter/2**. Dùng `MODE='resume'` chỉ khi đã đưa vào Kaggle Input gói Mini Evaluation SAFE của chính checkpoint đó.


In [ ]:
import sys,os,time,json,zipfile,subprocess,shutil
from pathlib import Path
SOURCE_COMMIT='7086a402b16dcacc46eed59339d101e3a9c73096'
SEED=20261111
MODE='new'  # new | resume (resume: gắn thêm Mini evaluation SAFE)
MAX_NEW_PAIRS=15  # tối đa 90 cặp: 30 video x 3 rates
SESSION_HOURS=8.0
REPO=Path('/kaggle/working/vvenc_rl_source')
TOOLS=Path('/kaggle/working/rl_rc_dot_tools')
TRAIN_ROOT=Path('/kaggle/working/mini_training_import')
EVAL_ROOT=Path('/kaggle/working/mini_test30_eval')
assert MODE in ('new','resume')
import torch
assert torch.cuda.is_available(),'Bật ít nhất một Kaggle GPU trước khi chạy'
print('MINI_TEST_CONFIG','30 videos','256 frames','50/120/200 kbps','max new pairs',MAX_NEW_PAIRS)


In [ ]:
def live(command,cwd=None):
    print('$',' '.join(map(str,command)),flush=True)
    proc=subprocess.Popen(list(map(str,command)),cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in proc.stdout:print(line,end='',flush=True)
        code=proc.wait()
    finally:
        if proc.poll() is None:proc.terminate();proc.wait()
    if code:raise RuntimeError('Command failed: see log above')

if not (REPO/'.git').exists():
    live(['git','clone','--branch','feature/rl-rc-dot-vvc','--depth','1',
          'https://github.com/youngan2810/vvenc_rl.git',REPO])
live(['git','fetch','--no-tags','--depth','1','origin',SOURCE_COMMIT],REPO)
live(['git','checkout','--detach',SOURCE_COMMIT],REPO)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==SOURCE_COMMIT
assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=REPO,text=True).strip()
# No VVC build until Test30 files are confirmed available.
live([sys.executable,'-c',"from scripts.rl_rc_dot.data import VideoCatalog; import json; c=VideoCatalog('/kaggle/input',mode='mini',seed=20261111); c.check_available(('test',)); print('MINI_TEST30_FILES_PASS',json.dumps(c.selected['test']))"],REPO)
live([sys.executable,'-m','scripts.rl_rc_dot.setup_kaggle','--repo',REPO,'--source-commit',SOURCE_COMMIT,
      '--output',TOOLS,'--jobs','2','--install-system'],REPO)
toolchain=json.loads((TOOLS/'TOOLCHAIN.json').read_text())


In [ ]:
sys.path.insert(0,str(REPO))
from scripts.rl_rc_dot.artifacts import import_run,read_checkpoint
from scripts.rl_rc_dot.data import source_fingerprint,digest

def inspect_export(path):
    with zipfile.ZipFile(path) as z:
        if 'session_payload.bin' in z.namelist():
            with z.open('session_payload.bin') as stream,zipfile.ZipFile(stream) as inner:
                return json.loads(inner.read('method.json')),json.loads(inner.read('progress.json'))
        return json.loads(z.read('method.json')),json.loads(z.read('progress.json'))

inputs=Path('/kaggle/input')
candidates=list(inputs.rglob('session_payload.bin'))
candidates += [p for p in inputs.rglob('*_KAGGLE_SAFE.zip') if p.is_file()]
valid=[]
for path in candidates:
    try:
        method,progress=inspect_export(path)
        if (method.get('dataset_mode')=='mini' and method.get('frames')==256
            and method.get('validation')=={'videos':4,'rates':[50,120],'frames':256}
            and method.get('source_fingerprint')==source_fingerprint()
            and method.get('encoder_sha256')==toolchain['encoder_sha256']
            and method.get('detector_weights_sha256')==digest(toolchain['weights'])
            and method.get('seed')==SEED and progress.get('best')):
            valid.append((progress['num_timesteps'],path))
    except (KeyError,ValueError,OSError,zipfile.BadZipFile):
        continue
assert valid,'Thiếu Mini Trial SAFE đúng source/config, hoặc checkpoint chưa có Validation best'
valid.sort(reverse=True,key=lambda x:x[0])
best_export=valid[0][1]
print('MINI_TRAIN_SOURCE',best_export,'STEPS',valid[0][0])
if not (TRAIN_ROOT/'progress.json').exists():
    import_run(best_export,TRAIN_ROOT)
trained_progress=json.loads((TRAIN_ROOT/'progress.json').read_text())
checkpoint_dir,checkpoint_meta=read_checkpoint(TRAIN_ROOT,trained_progress['best'])
CHECKPOINT=checkpoint_dir/'model.zip'
print('BEST_CHECKPOINT',CHECKPOINT,'VALIDATION_SCORE',trained_progress['best']['score'])


In [ ]:
# Paired evaluation: 15 new pairs per session. Export SAFE and resume for 90 total.
EVAL_RESUME=None
if MODE=='resume' and not (EVAL_ROOT/'evaluation.json').exists():
    all_inputs=list(Path('/kaggle/input').rglob('session_payload.bin'))
    all_inputs += list(Path('/kaggle/input').rglob('RL_RC_DOT_EVAL_*_KAGGLE_SAFE.zip'))
    for path in all_inputs:
        try:
            with zipfile.ZipFile(path) as archive:
                if 'session_payload.bin' in archive.namelist():
                    with archive.open('session_payload.bin') as stream,zipfile.ZipFile(stream) as inner:
                        schema=json.loads(inner.read('export_manifest.json')).get('source_schema')
                else:
                    schema=json.loads(archive.read('export_manifest.json')).get('source_schema')
            if schema=='vvenc-rcdot-evaluation/2':
                EVAL_RESUME=path
                break
        except (KeyError,OSError,ValueError,zipfile.BadZipFile):
            pass
    assert EVAL_RESUME,'MODE=resume: cần gắn Mini Evaluation SAFE từ phiên trước'
elif MODE=='new':
    assert not EVAL_ROOT.exists() or not any(EVAL_ROOT.iterdir()),'Evaluation output không rỗng; chọn resume'
cmd=[sys.executable,'-u','-m','scripts.rl_rc_dot.evaluate',
     '--input-root','/kaggle/input','--encoder',toolchain['encoder'],'--decoder',toolchain['decoder'],
     '--yolo-repo',toolchain['yolo_repo'],'--weights',toolchain['weights'],
     '--checkpoint',CHECKPOINT,'--output',EVAL_ROOT,'--dataset-mode','mini',
     '--rates','50,120,200','--frames','256','--seed',str(SEED),
     '--device','cuda:0','--encoder-threads','2','--max-pairs',str(MAX_NEW_PAIRS),
     '--max-hours',str(SESSION_HOURS)]
if EVAL_RESUME:cmd+=['--resume-from',str(EVAL_RESUME)]
live(cmd,REPO)


In [ ]:
from IPython.display import display,FileLink
summary=json.loads((EVAL_ROOT/'summary.json').read_text())
print('MINI_TEST30_SUMMARY',json.dumps(summary,indent=2,ensure_ascii=False))
exports=sorted((EVAL_ROOT/'exports').glob('*_KAGGLE_SAFE.zip'),key=lambda p:p.stat().st_mtime)
assert exports,'Không có evaluation SAFE; kiểm tra cell trên'
safe=Path('/kaggle/working/RL_RC_DOT_MINI_TEST30_LATEST_KAGGLE_SAFE.zip')
shutil.copyfile(exports[-1],safe)
print('TEST30_SAFE_EXPORT_READY',safe,'MB',round(safe.stat().st_size/1024**2,1))
os.chdir('/kaggle/working')
display(FileLink(safe.name))
print('Có thể phải chạy nhiều sessions để đạt đủ 90 cặp; không dùng Test30 để tune')
